# Nova AI — دمج النماذج (Mergekit) + تدريب خفيف تلقائي (Unsloth)

**التشغيل من الهاتف (Kaggle، وليس Colab):** دمج نموذجين بحجم 7B معاً
يحتاج فعلياً حوالي 30GB من الذاكرة — أثبتنا هذا مباشرة بالتجربة على
Colab المجاني (~12.7GB CPU RAM أو 15GB GPU VRAM فقط)، فتوقف الدمج
بنفاد الذاكرة في كل مرة. **Kaggle Notebooks** (مجاني بالكامل أيضاً)
يوفر **29GB من CPU RAM** — أقرب بكثير لما نحتاجه فعلياً.

## الإعداد لمرة واحدة فقط

**1) استورد الدفتر:** أنشئ حساباً على kaggle.com → **Create** → **New
Notebook** → **File** → **Import Notebook** → تبويب **GitHub** → الصق
رابط هذا الملف. من **Notebook options** → **Accelerator** فعّل **GPU
T4** (يطلب تفعيل رقم هاتفك مرة واحدة، مجاني).

**2) أضف 4 أسرار (Secrets) — من القائمة اليمنى: Add-ons → Secrets →
Add a new secret:**

| الاسم | القيمة |
|---|---|
| `HF_TOKEN` | توكن Hugging Face الخاص بك (Write access) |
| `HF_USERNAME` | اسم مستخدمك على Hugging Face |
| `SUPABASE_URL` | نفس القيمة المستخدمة على Render |
| `SUPABASE_SERVICE_ROLE_KEY` | نفس القيمة المستخدمة على Render |

هذه الأسرار لا تظهر أبداً داخل هذا الملف نفسه ولا تُحفظ معه — Kaggle
يخزّنها بشكل منفصل وآمن، وتبقى متاحة تلقائياً لأي تشغيل لاحق لهذا
الدفتر (يدوي أو مُجدوَل).

**3) فعّل التشغيل التلقائي الأسبوعي (بلا أي تدخل يدوي بعد اليوم):**
من القائمة اليمنى ابحث عن **"Schedule this notebook"** (أو من قائمة
الثلاث نقاط أعلى الدفتر) → اختر **Weekly** → احفظ. من الآن فصاعداً،
Kaggle نفسه يشغّل هذا الدفتر بالكامل من الأعلى للأسفل أسبوعياً تلقائياً
بدون فتحه أو الضغط على أي زر — يدمج، يتحقق من محادثات Nova الحقيقية
الجديدة، يحسّن النموذج إن توفرت بيانات كافية، ويرفع النتيجة لنفس مستودع
Hugging Face تلقائياً. هذا هو "التحديث والتدريب الذاتي" بمعناه الحقيقي
والصادق: أتمتة مُجدوَلة حقيقية عبر ميزة Kaggle نفسها، وليس نموذجاً "يعي
ذاته" أو يتصرف من تلقاء نفسه خارج هذا الجدول.

**ما يفعله هذا الدفتر في كل تشغيل:**
1. يدمج نموذجين مفتوحين (Qwen2.5-Coder-7B-Instruct + Qwen2.5-7B-Instruct) عبر خوارزمية TIES في Mergekit.
2. يجلب تلقائياً آخر المحادثات البرمجية الحقيقية من مستخدمي Nova (جدول NovaUsageLog في Supabase) ويحسّن النموذج بها عبر LoRA — يتخطى هذه الخطوة بأمان إن لم تتوفر بيانات كافية بعد، دون أي خطأ.
3. يرفع النتيجة إلى نفس مستودعك على Hugging Face Hub تلقائياً.

**أول مرة فقط:** بعد أول رفع، ضع اسم المستودع الذي يُطبع في الخلية 5
داخل `HF_SPECIALIST_MODEL_ID` على Render (Environment Variables) ثم
Manual Deploy. بعد ذلك، أي تحديث لاحق (يدوي أو مُجدوَل) يستبدل نفس
المستودع تلقائياً — لا حاجة لتكرار هذه الخطوة مجدداً.

In [ ]:
# الخلية 1 — تثبيت الأدوات (يأخذ بضع دقائق أول مرة فقط)
!pip install -q mergekit huggingface_hub
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# الخلية 2 — تسجيل الدخول لحسابك على Hugging Face (بلا أي تدخل يدوي)
#
# لجعل هذا الدفتر قابلاً للتشغيل التلقائي المُجدوَل بالكامل (Kaggle
# Schedule)، لا يمكن استخدام notebook_login() التفاعلي (يتوقف منتظراً
# لصق توكن يدوياً، وهذا يعطّل أي تشغيل مُجدوَل بلا إشراف). بدلاً من ذلك
# نقرأ التوكن من "Kaggle Secrets" — مكان آمن لتخزين المفاتيح لا يظهر
# أبداً في هذا الملف نفسه ولا يُرفع لأي مكان.
#
# مرة واحدة فقط، قبل أول تشغيل: من القائمة اليمنى في Kaggle اضغط
# Add-ons -> Secrets -> Add a new secret، وأضف:
#   الاسم: HF_TOKEN   —   القيمة: توكن Hugging Face الخاص بك (Write access)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("تم تسجيل الدخول إلى Hugging Face بنجاح")

In [ ]:
# الخلية 3 — إعداد دمج النماذج (Mergekit، خوارزمية TIES)
# غيّر هذين السطرين إذا أردت تجربة نموذجين آخرين متوافقين بنفس البنية والحجم.
merge_config = """
models:
  - model: Qwen/Qwen2.5-Coder-7B-Instruct
    parameters:
      weight: 0.6
      density: 0.6
  - model: Qwen/Qwen2.5-7B-Instruct
    parameters:
      weight: 0.4
      density: 0.6
merge_method: ties
base_model: Qwen/Qwen2.5-7B-Instruct
parameters:
  normalize: true
dtype: float16
"""
with open("merge_config.yaml", "w") as f:
    f.write(merge_config)
print("تمت كتابة merge_config.yaml")

In [ ]:
# الخلية 4 — تنفيذ الدمج الفعلي، عبر واجهة Python الداخلية لـ mergekit
# (وليس أمر !mergekit-yaml كعملية طرفية منفصلة).
#
# 1) هناك خلل حقيقي داخل مكتبة mergekit نفسها — عدة أصناف داخل ملف
# mergekit/architecture/base.py (مثل ConfiguredModuleArchitecture و
# ConfiguredModelArchitecture) تفشل تلقائياً في "حل" أنواعها المرجعية
# (torch، PretrainedConfig، وغيرها) عند إنشائها لأول مرة. الحل: نعيد بناء
# كل الأصناف في هذا الملف يدوياً قبل الدمج.
#
# 2) دمج 7B+7B يحتاج فعلياً ~30GB ذاكرة — أثبتنا هذا بتجربة مباشرة على
# Colab (فشل على GPU 15GB وعلى CPU RAM ~12.7GB كليهما، بغض النظر عن أي
# ضبط لخيارات mergekit). لهذا ننفّذ هذه الخلية الآن على Kaggle الذي يوفر
# 29GB من CPU RAM — cuda=False لأن المتوفر هنا هو ذاكرة معالج وفيرة، لا
# حاجة لإقحام GPU (15-16GB) في عملية تحتاج أكثر من سعته أصلاً.
import inspect
import torch
import yaml
from pydantic import BaseModel
from transformers import PretrainedConfig
import mergekit.architecture.base as _mkbase
_ns = dict(vars(_mkbase))
_ns.update({"PretrainedConfig": PretrainedConfig, "torch": torch})
for _name, _obj in list(vars(_mkbase).items()):
    if inspect.isclass(_obj) and issubclass(_obj, BaseModel):
        try:
            _obj.model_rebuild(force=True, _types_namespace=_ns)
        except Exception as _e:
            print("تعذر إعادة بناء", _name, "-", _e)
from mergekit.config import MergeConfiguration
from mergekit.merge import run_merge
from mergekit.options import MergeOptions
config_source = open("merge_config.yaml", "r", encoding="utf-8").read()
parsed_config = MergeConfiguration.model_validate(yaml.safe_load(config_source))
merge_options = MergeOptions(cuda=False, low_cpu_memory=False, lazy_unpickle=True)
run_merge(parsed_config, out_path="./nova-merged-model", options=merge_options, config_source=config_source)
print("اكتمل الدمج بنجاح")

In [ ]:
# الخلية 5 — رفع النموذج المدموج إلى حسابك (بلا أي تدخل يدوي)
#
# مرة واحدة فقط: أضف Kaggle Secret باسم HF_USERNAME وقيمته اسم مستخدمك
# الفعلي على Hugging Face (نفس مكان HF_TOKEN في الخلية 2 أعلاه).
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

HF_USERNAME = UserSecretsClient().get_secret("HF_USERNAME")
REPO_ID = f"{HF_USERNAME}/nova-coder-merge-7b"

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path="./nova-merged-model", repo_id=REPO_ID)
print(f"تم الرفع: https://huggingface.co/{REPO_ID}")
print("ضع هذا في HF_SPECIALIST_MODEL_ID داخل ai-system/.env:", REPO_ID)

---
## الخطوة الثانية (تلقائية بالكامل): تدريب خفيف LoRA على محادثات Nova الحقيقية

بدل كتابة أمثلة تدريب يدوياً، الخلايا التالية تجلب تلقائياً آخر
المحادثات البرمجية الحقيقية التي مرّت عبر Nova (من جدول NovaUsageLog
في Supabase) وتستخدمها كبيانات تدريب. إذا لم تتوفر بيانات كافية بعد،
تُتخطّى خطوات التدريب تلقائياً دون أي خطأ — الدمج والرفع في الخطوة
الأولى أعلاه يبقيان ناجحين بشكل مستقل تماماً على أي حال.


In [ ]:
# الخلية 6 — جلب بيانات تدريب حقيقية تلقائياً من محادثات Nova الفعلية
#
# بدل نسخة تدريب وهمية تُكتب يدوياً، هذا يقرأ آخر المحادثات البرمجية
# (queryType = CODE) الحقيقية التي مرّت عبر Nova من جدول NovaUsageLog
# في Supabase مباشرة عبر REST API — بلا أي تدخل يدوي، وبلا كتابة أي
# مفتاح سري داخل هذا الملف (يُقرأ من Kaggle Secrets فقط، تماماً كخطوة
# تسجيل الدخول في الخلية 2). هذا هو "بنك المعرفة" الذي يكبر تلقائياً مع
# كل استخدام حقيقي للبوت، ويُستخدم هنا لتحسين النموذج المتخصص أسبوعياً.
#
# مرة واحدة فقط: أضف Kaggle Secret باسم SUPABASE_URL وآخر باسم
# SUPABASE_SERVICE_ROLE_KEY (نفس القيمتين المستخدمتين على Render).
import requests
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
_SUPABASE_URL = _secrets.get_secret("SUPABASE_URL")
_SUPABASE_KEY = _secrets.get_secret("SUPABASE_SERVICE_ROLE_KEY")

_resp = requests.get(
    f"{_SUPABASE_URL}/rest/v1/NovaUsageLog",
    headers={"apikey": _SUPABASE_KEY, "Authorization": f"Bearer {_SUPABASE_KEY}"},
    params={
        "select": "message,answer",
        "queryType": "eq.CODE",
        "message": "not.is.null",
        "answer": "not.is.null",
        "order": "created_at.desc",
        "limit": "300",
    },
    timeout=30,
)
_rows = _resp.json() if _resp.ok else []
training_data = [{"question": r["message"], "answer": r["answer"]} for r in _rows if r.get("message") and r.get("answer")]
HAVE_TRAINING_DATA = len(training_data) >= 5  # حد أدنى بسيط لتفادي تدريب على عدد ضئيل جداً
print(f"عدد أمثلة التدريب الحقيقية المتوفرة: {len(training_data)}")
if not HAVE_TRAINING_DATA:
    print("لا توجد بيانات كافية بعد — سيتم تخطي خطوة التدريب هذه المرة (الدمج والرفع أعلاه تم بنجاح بالفعل).")

In [ ]:
# الخلية 7 — تحميل النموذج المدموج عبر Unsloth للتدريب السريع (4-bit QLoRA)
# يعمل فقط إن توفرت بيانات تدريب حقيقية كافية (الخلية السابقة).
#
# "ذاكرة إضافية فوق سعة الـ GPU" — تفعيل تلقائي بالكامل: يحاول الكود
# التحميل العادي داخل الـ GPU أولاً (الأسرع)، وفقط عندما يفشل فعلاً
# بسبب امتلاء الذاكرة (CUDA Out Of Memory) يعيد المحاولة تلقائياً
# موزّعاً الطبقات بين الـ GPU وذاكرة CPU RAM والقرص (Offloading).
if HAVE_TRAINING_DATA:
    import torch
    from unsloth import FastLanguageModel

    MAX_GPU_MEMORY = "15GiB"   # هامش أمان عن حد T4 على Kaggle (~16GB)
    MAX_CPU_MEMORY = "28GiB"   # هامش أمان عن حد ذاكرة Kaggle المجانية (~29GB)

    def _load_model(offload: bool):
        extra_kwargs = {}
        if offload:
            extra_kwargs = {
                "device_map": "auto",
                "max_memory": {0: MAX_GPU_MEMORY, "cpu": MAX_CPU_MEMORY},
                "offload_folder": "./offload",
            }
        return FastLanguageModel.from_pretrained(
            model_name=REPO_ID,
            max_seq_length=2048,
            load_in_4bit=True,
            **extra_kwargs,
        )

    try:
        model, tokenizer = _load_model(offload=False)
        print("تم التحميل مباشرة داخل الـ GPU — النموذج يتسع ضمن الذاكرة المتوفرة.")
    except torch.cuda.OutOfMemoryError:
        print("النموذج أكبر من سعة الـ GPU — تفعيل تمديد الذاكرة (CPU/Disk Offload) تلقائياً...")
        torch.cuda.empty_cache()
        model, tokenizer = _load_model(offload=True)

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
else:
    print("تخطي تحميل النموذج للتدريب — لا توجد بيانات كافية بعد.")

In [ ]:
# الخلية 8 — تجهيز البيانات والتدريب الفعلي (يعمل فقط إن توفرت بيانات كافية)
if HAVE_TRAINING_DATA:
    from datasets import Dataset

    def format_example(ex):
        return {"text": f"### سؤال:\n{ex['question']}\n### إجابة:\n{ex['answer']}"}

    dataset = Dataset.from_list(training_data).map(format_example)

    from trl import SFTTrainer
    from transformers import TrainingArguments

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=2048,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            num_train_epochs=1,
            learning_rate=2e-4,
            output_dir="./nova-lora-out",
            logging_steps=1,
        ),
    )
    trainer.train()
else:
    print("تخطي التدريب — لا توجد بيانات كافية بعد.")

In [ ]:
# الخلية 9 — رفع النسخة المُحسَّنة (يعمل فقط إن جرى تدريب فعلي أعلاه)
if HAVE_TRAINING_DATA:
    model.push_to_hub_merged(REPO_ID, tokenizer, save_method="merged_16bit")
    print(f"تم تحديث النموذج المُحسَّن على: https://huggingface.co/{REPO_ID}")
else:
    print("تخطي الرفع النهائي — لم يُجرَ تدريب هذه المرة.")